In [1]:
!pip install openai pandas pillow tqdm

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [7]:
import os
import shutil

from google.colab import files

UPLOAD_DIR = "/content/generated_wheels"
os.makedirs(UPLOAD_DIR, exist_ok=True)

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, os.path.join(UPLOAD_DIR, filename))

print("Uploaded files:")
print(os.listdir(UPLOAD_DIR))

Saving Снимок экрана 2026-05-21 142637.png to Снимок экрана 2026-05-21 142637.png
Uploaded files:
['Снимок экрана 2026-05-21 142637.png']


In [4]:
VALIDATION_PROMPT = """
You are an expert in automotive visual inspection and wheel fitment plausibility.

Analyze the provided image of a car with generated or modified wheel disks/rims.

Evaluate the image according to the following criteria:

1. Visual attractiveness:
   - Do the wheels look aesthetically pleasing?
   - Do they improve the overall appearance of the car?

2. Style consistency:
   - Do the wheels match the car type, body shape, and visual identity?
   - Are they too sporty, too luxurious, too simple, or inconsistent?

3. Physical realism:
   - Are the wheels circular and not distorted?
   - Are they correctly positioned inside the wheel arches?
   - Are the front and rear wheels consistent?
   - Are there visible generation artifacts?

4. Approximate fitment plausibility:
   - Do the wheels look too large or too small?
   - Is the tire thickness visually plausible?
   - Do the wheels intersect with the fender, body, or road?
   - Does the wheel placement look physically possible?

Important limitation:
You cannot measure exact technical parameters such as bolt pattern, offset, center bore, or exact rim diameter.
Only evaluate visual and approximate physical plausibility.

Return ONLY valid JSON with the following structure:

{
  "visual_attractiveness_score": 1,
  "style_consistency_score": 1,
  "physical_realism_score": 1,
  "fitment_plausibility_score": 1,
  "detected_problems": [],
  "overall_score": 1,
  "decision": "accept",
  "short_explanation": "..."
}

Scoring:
1 = very poor
2 = poor
3 = acceptable but needs review
4 = good
5 = excellent

Decision rules:
- "accept" if overall quality is good and no serious problems are detected.
- "needs_review" if the result is acceptable but has minor issues.
- "reject" if there are serious visual, geometric, or physical problems.
"""

In [5]:
import base64
import json

from openai import OpenAI

client = OpenAI()


def encode_image_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def validate_wheel_image(image_path, model="gpt-4.1-mini"):
    image_base64 = encode_image_to_base64(image_path)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": VALIDATION_PROMPT},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image_base64}"},
                ],
            }
        ],
        temperature=0,
    )

    text = response.output_text.strip()

    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        result = {
            "visual_attractiveness_score": None,
            "style_consistency_score": None,
            "physical_realism_score": None,
            "fitment_plausibility_score": None,
            "detected_problems": ["JSON parsing failed"],
            "overall_score": None,
            "decision": "needs_review",
            "short_explanation": text,
        }

    result["image_name"] = os.path.basename(image_path)
    return result

In [ ]:
import pandas as pd
from tqdm import tqdm

image_extensions = (".jpg", ".jpeg", ".png", ".webp")

image_paths = [
    os.path.join(UPLOAD_DIR, f)
    for f in os.listdir(UPLOAD_DIR)
    if f.lower().endswith(image_extensions)
]

results = []

for image_path in tqdm(image_paths):
    result = validate_wheel_image(image_path)
    results.append(result)

df = pd.DataFrame(results)

cols = ["image_name"] + [c for c in df.columns if c != "image_name"]
df = df[cols]

df

In [ ]:
import pandas as pd
from tqdm import tqdm

image_extensions = (".jpg", ".jpeg", ".png", ".webp")

image_paths = [
    os.path.join(UPLOAD_DIR, f)
    for f in os.listdir(UPLOAD_DIR)
    if f.lower().endswith(image_extensions)
]

results = []

for image_path in tqdm(image_paths):
    result = validate_wheel_image(image_path)
    results.append(result)

df = pd.DataFrame(results)

cols = ["image_name"] + [c for c in df.columns if c != "image_name"]
df = df[cols]

df

In [ ]:
OUTPUT_CSV = "/content/wheel_vlm_validation_results.csv"

df.to_csv(OUTPUT_CSV, index=False)

files.download(OUTPUT_CSV)